In [8]:
import streamlit as st
import boto3
import json
import os

# Load endpoint name
if os.path.exists("endpoint.txt"):
    with open("endpoint.txt", "r") as f:
        ENDPOINT_NAME = f.read().strip()
else:
    # fallback: get latest endpoint from SageMaker
    sm = boto3.client("sagemaker")
    endpoints = sm.list_endpoints(SortBy="CreationTime", SortOrder="Descending")
    ENDPOINT_NAME = endpoints["Endpoints"][0]["EndpointName"]

runtime = boto3.client("sagemaker-runtime")

st.set_page_config(page_title="Credit Risk Predictor", layout="centered")
st.title("💳 Credit Risk Prediction")
st.write("Enter applicant details to predict the probability of loan default.")

with st.form("credit_form"):
    age = st.number_input("Age", min_value=18, max_value=100, value=30)
    income = st.number_input("Annual Income ($)", min_value=5000, max_value=500000, value=50000)
    loan_amount = st.number_input("Loan Amount ($)", min_value=1000, max_value=200000, value=15000)
    submitted = st.form_submit_button("Predict Risk")

if submitted:
    features = [age, income, loan_amount]
    payload = json.dumps({"features": features})

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=payload
    )
    result = json.loads(response["Body"].read().decode())

    prediction = result["prediction"][0]

    st.subheader("📊 Prediction Result")
    if "probabilities" in result:
        probabilities = result["probabilities"][0]
        st.write(f"**Prediction:** {'Default (1)' if prediction == 1 else 'No Default (0)'}")
        st.write(f"**Probabilities:** Default = {probabilities[1]:.2f}, No Default = {probabilities[0]:.2f}")
        st.progress(probabilities[1])
        if prediction == 1:
            st.error("⚠️ High Risk: Loan likely to default!")
        else:
            st.success("✅ Low Risk: Loan unlikely to default.")
    else:
        st.write(f"**Prediction:** {'Default (1)' if prediction == 1 else 'No Default (0)'}")


2025-08-07 01:55:43.369 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 01:55:43.370 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 01:55:43.694 
  command:

    streamlit run /home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-08-07 01:55:43.695 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 01:55:43.696 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 01:55:43.697 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-08-07 01:55:43.697 Thread 'MainThread': missing ScriptRunContext! This war

In [7]:
pip install streamlit

  Using cached gitpython-3.1.45-py3-none-any.whl.metadata (13 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached smmap-5.0.2-py3-none-any.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 62.2 MB/s eta 0:00:00
Using cached gitpython-3.1.45-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.5 MB/s eta 0:00:00
Using cached smmap-5.0.2-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [streamlit]/9 [streamlit]]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!streamlit run credit_app.py --server.port 8501 --server.headless true --server.address 0.0.0.0




  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

